# statwarrant vs language models on MedLitSpin — the full experiment, step by step

This notebook re-runs every number in Section 7 of the paper from the files in this folder.
Run it top to bottom (**Kernel → Restart & Run All**). Nothing is hidden in other scripts.

**What is recomputed here, live:** the extraction checks, the conversion to z, statwarrant's rule
checks on all 60 conclusions, the combined route (model reads, statwarrant decides), all accuracies,
confidence intervals, McNemar tests and the numerical comparison with SciPy and statsmodels.

**What is loaded, not recomputed:** the language models' answers. The 22 published models come from
the MedLitSpin repository (Yun et al., CHIL 2025). The Claude answers were recorded on 27 November 2026
and are stored in `llm_answers/`. You can read every one of them below. Re-running Claude yourself
needs an Anthropic API key (optional cell at the end).

Data: MedLitSpin (github.com/hyesunyun/MedLitSpin), Apache-2.0, 60 abstracts of 30 cancer RCTs with
non-significant primary outcomes: 30 with spin (Boutron et al., SPIIN 2014) and 30 neutral rewrites.

## 0. Setup
The paper reports statwarrant **0.6.0**. With 0.6.1 the MedLitSpin results are identical (checked).

In [1]:
# !pip install statwarrant==0.6.0 pandas scipy statsmodels   # uncomment if needed
from IPython.display import display
import json, math, re, csv, glob, os, random
import pandas as pd
from scipy.stats import norm, binomtest
import statwarrant as sw
from statwarrant.llm import CallableLLM
print('statwarrant', sw.__version__)
pd.set_option('display.max_colwidth', 200); pd.set_option('display.width', 250)

statwarrant 0.6.0


## 1. The data and the labels
The label (`abstract_type`) comes from the dataset, not from us. statwarrant never sees it.

In [2]:
d = pd.read_csv('data/spin_and_no_spin_abstracts.csv')
print(d.shape); print(d.abstract_type.value_counts())
d[['id', 'PMID', 'abstract_type', 'title']].head(6)

(60, 6)
abstract_type
spin       30
no_spin    30
Name: count, dtype: int64


,id,PMID,abstract_type,title
0,1,17173959,spin,Effective cervical neoplasia detection with a novel optical detection system: a randomized trial
1,2,17173959,no_spin,Effective cervical neoplasia detection with a novel optical detection system: a randomized trial
2,3,17530429,spin,"Phase III randomized multicenter study on the effects of adjuvant CMF in patients with node-negative, rapidly proliferating breast cancer: twelve-year results and retrospective subgroup analysis"
3,4,17530429,no_spin,"Phase III randomized multicenter study on the effects of adjuvant CMF in patients with node-negative, rapidly proliferating breast cancer: twelve-year results and retrospective subgroup analysis"
4,5,21471562,spin,"Impact of a novel nurse-led prechemotherapy education intervention (ChemoEd) on patient distress, symptom burden, and treatment-related information and support needs: results from a randomised, co..."
5,6,21471562,no_spin,"Impact of a novel nurse-led prechemotherapy education intervention (ChemoEd) on patient distress, symptom burden, and treatment-related information and support needs: results from a randomised, co..."


In [3]:
# One trial in both versions (read them):
pm = d.PMID.iloc[0]
for _, r in d[d.PMID == pm].iterrows():
    print(f'===== {r.abstract_type.upper()} (id {r.id}) =====\n{r.abstract}\n')

===== SPIN (id 1) =====
Objective:
To assess whether the use of intervention A as an adjunct to colposcopy increases the detection of biopsy confirmed CIN 2, 3 (high grade cervical intraepithelial neoplasia).
Methods:
This is a multicenter two‐arm randomized trial comparing colposcopy alone with colposcopy plus intervention A. Patients were recruited from 13 colposcopy clinics in a variety of practice settings. 2299 women referred for the evaluation of an abnormal cervical cytology were randomized with stratification by cytology; subsequently 113 women were excluded for a variety of reasons. The main study outcomes were differences in true‐positive rates (CIN 2, 3 and cancer identified) and false‐positive rates between the study arms.
Results:
The true‐positive (TP) rates were 14.4% vs. 11.4% (p=0.035, one‐sided) for the combined colposcopy and intervention A arm compared to colposcopy‐only arm, respectively, in women with either an atypical squamous cell (ASC) or low‐grade squamous in

## 2. The protocol (written before statwarrant was run)

In [4]:
print(open('data/PROTOCOL.md').read())

# Experiment: statwarrant 0.6.0 vs LLMs on MedLitSpin (written before any statwarrant run)

Data: MedLitSpin (Yun et al., CHIL 2025; Apache-2.0), data/spin_and_no_spin_abstracts.csv.
60 abstracts of 30 cancer RCTs with non-significant primary outcomes: 30 with spin (Boutron et al. 2014,
SPIIN) and 30 neutral rewrites. Label = abstract_type. Labels are external to this project.

Primary result per trial: extracted from the NEUTRAL abstract only (numbers are shared by both versions),
before running statwarrant, into extraction.csv. Represented as a signed z statistic (estimate = z,
SE = 1, normal reference): positive = favours treatment A. z comes from the reported CI when a ratio
with CI is given (z = -ln(ratio)/SE_log for hazard ratios; +ln(OR)/SE_log for odds ratios of a good
outcome), else from the reported two-sided p and the direction of the reported summaries, else from
the reported counts (two-proportion z). No SESOI is declared (none is reported).

Text checked: the abstract's c

## 3. Primary result of each trial
Extracted from the **neutral** abstract (both versions share the numbers). `kind`: `hr`/`or` = ratio with 95% CI,
`p` = only a p-value (with the direction of the summaries), `counts` = responders. Read the `note` column against the abstract.

In [5]:
ex = pd.read_csv('data/extraction.csv', dtype=str).fillna('')
ex

,pmid,outcome,comparator,kind,value,lo,hi,p,direction,note
0,9093724,survival,comparator B,p,,,,0.54,A,median 18 vs 17 months
1,10547391,occurrence of contralateral breast cancer,the control arm,hr,0.92,0.66,1.29,0.642,,events
2,11261827,response rate,comparator B,counts,68/106,58/111,,,A,response 64% vs 52%; no p reported
3,10637238,survival,chemotherapy alone,p,,,,0.834,C,median 30.0 control vs 29.3 A
4,12177098,time to progression,comparator B,hr,0.92,0.74,1.14,0.43,,
5,15947110,overall response rate,epirubicin alone,counts,87/106,80/105,,,A,82% vs 76%; counts rounded from %
6,16148021,disease-free survival,group B,p,,,,0.55,A,3-year DFS 80% vs 77%
7,16314619,survival,the control arm,p,,,,0.16,A,MST 5.4 vs 4.4; HR 0.87 without CI
8,16504757,disease-free survival,comparator B,p,,,,0.22,A,5-year DFS 83% vs 79%
9,17134892,survival,no further treatment,hr,0.97,0.79,1.18,0.7377,,


### 3a. Independent second extraction (by another AI agent, blind to the first) and agreement

In [6]:
ex2 = pd.read_csv('data/extraction_second.csv', dtype=str).fillna('')
ex2['pmid'] = ex2['pmid'].str.strip()
def direction(r):
    if r['kind'] == 'hr': return 'A' if float(r['value']) < 1 else 'comparator'
    if r['kind'] == 'or': return 'A' if float(r['value']) > 1 else 'comparator'
    return 'A' if r['direction'] == 'A' else 'comparator'
cmp = ex.assign(dir1=ex.apply(direction, axis=1)).merge(ex2, on='pmid', suffixes=('_1', '_2'))
cmp['dir2'] = cmp['favours'].str.strip()
cmp['same_direction'] = cmp['dir1'] == cmp['dir2']
cmp['same_p'] = cmp['p_1'] == cmp['p_2'].str.strip()
print('outcome pairs compared:', len(cmp), '| direction agrees:', cmp.same_direction.sum(), '/', len(cmp))
cmp[['pmid', 'outcome_1', 'outcome_2', 'dir1', 'dir2', 'p_1', 'p_2', 'same_direction']]

outcome pairs compared: 30 | direction agrees: 29 / 30


,pmid,outcome_1,outcome_2,dir1,dir2,p_1,p_2,same_direction
0,9093724,survival,overall survival,A,A,0.54,0.54,True
1,10547391,occurrence of contralateral breast cancer,contralateral breast cancer incidence (first of the combined primary),A,A,0.642,0.642,True
2,11261827,response rate,response rate (first listed main end point),A,A,,none,True
3,10637238,survival,median survival (first reported; no primary named),comparator,comparator,0.834,0.834,True
4,12177098,time to progression,time to progression,A,A,0.43,0.43,True
5,15947110,overall response rate,overall response rate,A,A,,none,True
6,16148021,disease-free survival,disease-free survival,A,A,0.55,0.55,True
7,16314619,survival,survival,A,A,0.16,0.16,True
8,16504757,disease-free survival,disease-free survival,A,A,0.22,0.22,True
9,17134892,survival,5-year survival,A,A,0.7377,0.7377,True


### 3b. Conversion to a signed z (positive = favours treatment A)
From the CI of a ratio: z = −ln(HR)/SE (or +ln(OR)/SE), SE = (ln hi − ln lo)/(2·1.96). From a p-value: z = Φ⁻¹(1 − p/2) with the sign of the direction. From counts: two-proportion z.

In [7]:
def zstat(r):
    k = r['kind']
    if k in ('hr', 'or'):
        v, lo, hi = float(r['value']), float(r['lo']), float(r['hi'])
        se = (math.log(hi) - math.log(lo)) / (2 * norm.isf(0.025))
        z = math.log(v) / se
        return -z if k == 'hr' else z
    if k == 'p':
        z = norm.isf(float(r['p']) / 2)
        return z if r['direction'] == 'A' else -z
    (a, na), (b, nb) = [tuple(map(int, s.split('/'))) for s in (r['value'], r['lo'])]
    p1, p2, p = a / na, b / nb, (a + b) / (na + nb)
    return (p1 - p2) / math.sqrt(p * (1 - p) * (1 / na + 1 / nb))

ex['z'] = ex.apply(zstat, axis=1)
ex['p_from_z'] = 2 * norm.sf(ex['z'].abs())
ex[['pmid', 'outcome', 'kind', 'value', 'lo', 'hi', 'p', 'z', 'p_from_z']].round(4)

,pmid,outcome,kind,value,lo,hi,p,z,p_from_z
0,9093724,survival,p,,,,0.54,0.6128,0.5400
1,10547391,occurrence of contralateral breast cancer,hr,0.92,0.66,1.29,0.642,0.4877,0.6257
2,11261827,response rate,counts,68/106,58/111,,,1.7756,0.0758
3,10637238,survival,p,,,,0.834,-0.2096,0.8340
4,12177098,time to progression,hr,0.92,0.74,1.14,0.43,0.7564,0.4494
5,15947110,overall response rate,counts,87/106,80/105,,,1.0521,0.2928
6,16148021,disease-free survival,p,,,,0.55,0.5978,0.5500
7,16314619,survival,p,,,,0.16,1.4051,0.1600
8,16504757,disease-free survival,p,,,,0.22,1.2265,0.2200
9,17134892,survival,hr,0.97,0.79,1.18,0.7377,0.2976,0.7660


## 4. Numbers: statwarrant vs SciPy vs statsmodels (condition D)

In [8]:
from statsmodels.stats.weightstats import _zstat_generic2
EX = {int(r['pmid']): r for _, r in ex.iterrows()}

def result_for(r):
    plan = sw.declare(outcome=r['outcome'], contrast=f"treatment A minus {r['comparator']}", better='higher',
                      design='randomised', units='z (positive favours treatment A)')
    return plan.analyze(estimate=float(r['z']), se=1.0)

rows = []
for pm, r in EX.items():
    p_sw = result_for(r).evidence.p_zero
    p_scipy = 2 * norm.sf(abs(r['z']))
    p_sm = _zstat_generic2(r['z'], 1.0, 'two-sided')[1]
    rows.append(dict(pmid=pm, p_statwarrant=p_sw, p_scipy=p_scipy, p_statsmodels=p_sm, p_printed=r['p']))
num = pd.DataFrame(rows)
print('max |statwarrant - scipy|      =', (num.p_statwarrant - num.p_scipy).abs().max())
print('max |statwarrant - statsmodels| =', (num.p_statwarrant - num.p_statsmodels).abs().max())
print('all primary results non-significant:', (num.p_statwarrant > 0.05).all())
num.round(4)

max |statwarrant - scipy|      = 0.0
max |statwarrant - statsmodels| = 0.0
all primary results non-significant: True


,pmid,p_statwarrant,p_scipy,p_statsmodels,p_printed
0,9093724,0.5400,0.5400,0.5400,0.54
1,10547391,0.6257,0.6257,0.6257,0.642
2,11261827,0.0758,0.0758,0.0758,
3,10637238,0.8340,0.8340,0.8340,0.834
4,12177098,0.4494,0.4494,0.4494,0.43
5,15947110,0.2928,0.2928,0.2928,
6,16148021,0.5500,0.5500,0.5500,0.55
7,16314619,0.1600,0.1600,0.1600,0.16
8,16504757,0.2200,0.2200,0.2200,0.22
9,17134892,0.7660,0.7660,0.7660,0.7377


## 5. The text checked: each abstract's conclusion
Rule (same for both versions): text after the last "Conclusion(s):", "Interpretation:" or "In conclusion,"; otherwise the last two sentences.

In [9]:
def split_sentences(t):
    return [s for s in re.split(r'(?<=[.!?])\s+(?=[A-Z])', t.strip()) if s]

def conclusion(abstract):
    a = abstract.replace('\u2010', '-').replace('\u2011', '-')
    m = list(re.finditer(r'(Conclusions?\s*:|Interpretation\s*:|In conclusion,)', a))
    if m:
        return a[m[-1].end():].strip().replace('\n', ' ')
    return ' '.join(split_sentences(a.replace('\n', ' '))[-2:])

d['conclusion'] = d.abstract.apply(conclusion)
d[['id', 'PMID', 'abstract_type', 'conclusion']].sort_values(['PMID', 'abstract_type'])

,id,PMID,abstract_type,conclusion
13,14,9093724,no_spin,Treatment A was not more effective than comparator B in patients with metastatic breast cancer.
12,13,9093724,spin,"This trial shows that treatment A can be administered for repeated cycles without bone marrow support with increased, though acceptable, toxicity and with a significant increase of antitumor effec..."
57,58,10547391,no_spin,Treatment A was not more effective than no treatment in preventing a second breast malignancy in women with breast cancer.
56,57,10547391,spin,"Treatment A of women with breast cancer for 5 years appears to have no statistically significant effect on the incidence of second breast malignancies overall, although a possible benefit was dete..."
45,46,10637238,no_spin,"In patients with metastatic breast cancer, treatment A with chemotherapy was not more effective than chemotherapy alone."
44,45,10637238,spin,"In patients with potentially hormone-sensitive metastatic breast cancer, treatment A + chemotherapy prolongs TTF for ER-positive patients without improving overall survival."
27,28,11261827,no_spin,Treatment A was not more effective than comparator B in postmenopausal women with advanced breast cancer.
26,27,11261827,spin,"Our data suggest that treatment A is an efficient and well-tolerated agent for the therapy of postmenopausal women with hormonal positive receptors advanced breast cancer, and must be considered a..."
33,34,12177098,no_spin,Treatment A was not more effective than comparator B for postmenopausal women with advanced breast cancer whose disease progresses on tamoxifen therapy.
32,33,12177098,spin,"Treatment A was at least as effective as comparator B, with efficacy end points slightly favoring treatment A. Treatment A represents an additional treatment option for postmenopausal women with a..."


## 6. Condition A — statwarrant rules alone (recomputed now)
Spin is flagged when the conclusion has at least one **conflict**.

In [10]:
rule_rows = []
for _, a in d.iterrows():
    res = result_for(EX[int(a.PMID)])
    c = res.check(a.conclusion, record=False)
    rule_rows.append(dict(id=int(a.id), PMID=int(a.PMID), label=a.abstract_type, level=c.level or 'none',
                          conflicts='; '.join(f'{p.rule}: "{p.text}"' for p in c.parts if p.status == 'conflict'),
                          to_verify='; '.join(p.rule for p in c.parts if p.status == 'check'),
                          conclusion=a.conclusion))
rules = pd.DataFrame(rule_rows).sort_values(['PMID', 'label'])
rules['flag'] = rules.level == 'conflict'
pd.crosstab(rules.label, rules.flag, margins=True)

flag,False,True,All
label,,,
no_spin,28,2,30
spin,11,19,30
All,39,21,60


In [11]:
# Every decision, with the words that caused the conflict:
rules[['PMID', 'label', 'flag', 'conflicts', 'to_verify', 'conclusion']]

,PMID,label,flag,conflicts,to_verify,conclusion
13,9093724,no_spin,False,,no-effect,Treatment A was not more effective than comparator B in patients with metastatic breast cancer.
12,9093724,spin,True,"significance: ""significant""",unattached-negation; direction,"This trial shows that treatment A can be administered for repeated cycles without bone marrow support with increased, though acceptable, toxicity and with a significant increase of antitumor effec..."
57,10547391,no_spin,False,,no-effect; unattached-negation; subgroup,Treatment A was not more effective than no treatment in preventing a second breast malignancy in women with breast cancer.
56,10547391,spin,False,,nonsignificance; direction,"Treatment A of women with breast cancer for 5 years appears to have no statistically significant effect on the incidence of second breast malignancies overall, although a possible benefit was dete..."
45,10637238,no_spin,False,,no-effect,"In patients with metastatic breast cancer, treatment A with chemotherapy was not more effective than chemotherapy alone."
44,10637238,spin,False,,direction; unattached-negation,"In patients with potentially hormone-sensitive metastatic breast cancer, treatment A + chemotherapy prolongs TTF for ER-positive patients without improving overall survival."
27,11261827,no_spin,False,,no-effect,Treatment A was not more effective than comparator B in postmenopausal women with advanced breast cancer.
26,11261827,spin,True,"direction: ""positive""",,"Our data suggest that treatment A is an efficient and well-tolerated agent for the therapy of postmenopausal women with hormonal positive receptors advanced breast cancer, and must be considered a..."
33,12177098,no_spin,False,,no-effect,Treatment A was not more effective than comparator B for postmenopausal women with advanced breast cancer whose disease progresses on tamoxifen therapy.
32,12177098,spin,True,"equivalence: ""as effective as""; small: ""slightly""",,"Treatment A was at least as effective as comparator B, with efficacy end points slightly favoring treatment A. Treatment A represents an additional treatment option for postmenopausal women with a..."


In [12]:
# Full statwarrant output for one spun conclusion (change the index to read others):
a = d[d.abstract_type == 'spin'].iloc[0]
print(a.conclusion, '\n')
print(result_for(EX[int(a.PMID)]).check(a.conclusion, record=False))

Combining intervention A with colposcopy provides a clinically meaningful increase in the detection of CIN 2, 3 in women referred for the evaluation of mildly abnormal cytology results. 

[1 conflict, 3 to verify] Combining intervention A with colposcopy provides a clinically meaningful increase in the detection of CIN 2, 3 in women referred for the evaluation of mildly abnormal cytology results.
  Conflicts with the result:
  - direction              "increase": The direction of the effect is not established: 95% CI [-0.495, 3.42] (endpoints rounded for display) includes zero. This holds whichever way the comparison runs.
    Check: Check the direction against the interval.
  Not verified (statwarrant could not check these; review them by hand):
  - importance             "clinically meaningful": Not checked against the result: this importance claim depends on which way the comparison runs, and the wording could not be tied to the declared contrast (treatment A minus colposcopy alone)

## 7. Condition B — language models alone (recorded answers)
### 7a. The 22 published models (Yun et al. 2025), per abstract

In [13]:
label = {int(r.id): r.abstract_type == 'spin' for _, r in d.iterrows()}
IDS = sorted(label)
preds = {'statwarrant rules (conflict)': {r.id: r.flag for r in rules.itertuples()}}
published = {}
for f in sorted(glob.glob('published_llm_outputs/*.csv')):
    t = pd.read_csv(f)
    published[os.path.basename(f)[:-4] + ' (published)'] = {int(r.id): str(r.model_answer).strip().lower().startswith('yes')
                                                             for _, r in t.iterrows()}
print(len(published), 'published models')
pd.read_csv('published_llm_outputs/gpt4o.csv').head()

22 published models


,id,PMID,abstract_type,model_answer
0,58,10547391,no_spin,no
1,57,10547391,spin,yes
2,46,10637238,no_spin,no
3,45,10637238,spin,yes
4,27,11261827,spin,yes


### 7b. Claude, this study (recorded 27 Nov 2026)
Abstracts were given coded (`Xnn`) and shuffled; `data/blind_key.json` maps codes to ids.
* **full abstract, split**: the two versions of a trial were never in the same context (main result).
* **full abstract, one context**: all 60 together (sensitivity check).
* **same inputs as statwarrant**: only the conclusion + the primary result (`llm_answers/aligned_half*.txt` shows exactly what the model saw).

In [14]:
key = {int(k): v for k, v in json.load(open('data/blind_key.json')).items()}
inv = {v: k for k, v in key.items()}
def load_answers(files):
    out = {}
    for f in files:
        for r in csv.DictReader(open(f)):
            out[inv[r['code'].strip()]] = r['answer'].strip().lower().startswith('y')
    assert len(out) == 60, f
    return out
for m in ('haiku', 'sonnet', 'opus'):
    preds[f'Claude {m}, full abstract'] = load_answers([f'llm_answers/split_{m}_half{h}.csv' for h in (1, 2)])
    preds[f'Claude {m}, full abstract, one context'] = load_answers([f'llm_answers/llm_alone_{m}.csv'])
    preds[f'Claude {m}, same inputs as statwarrant'] = load_answers([f'llm_answers/aligned_{m}_half{h}.csv' for h in (1, 2)])
print(open('llm_answers/aligned_half1.txt').read()[:1500])

### X04
Primary outcome: sleep quality; comparison: treatment A versus placebo. Two-sided p = 0.7; the point estimate favours the comparator. The primary result is not statistically significant.
Conclusion: However, exploratory analyses revealed improvement in some secondary outcomes, such as fatigue. Further research with treatment A exploring physiologic effects in oncology symptom management may be warranted.

### X43
Primary outcome: survival; comparison: treatment A versus chemotherapy alone. Two-sided p = 0.834; the point estimate favours the comparator. The primary result is not statistically significant.
Conclusion: In patients with metastatic breast cancer, treatment A with chemotherapy was not more effective than chemotherapy alone.

### X06
Primary outcome: overall response rate; comparison: treatment A versus CAF + placebo. Two-sided p = 0.077; the point estimate favours treatment A. The primary result is not statistically significant.
Conclusion: Treatment A + CAF was not 

## 8. Condition C — model reads, statwarrant decides (recomputed from recorded readings)
The model only listed the conclusion's claims as JSON (it never saw the numbers). statwarrant judges them **now**.
Spin is flagged when an asserted claim is *contradicted* or *not established*; a variant also adds rule conflicts.

In [15]:
reader_detail = {}
for m in ('haiku', 'sonnet'):
    ans = json.load(open(f'llm_answers/reader_{m}.json'))
    flag, ok, item_err = {}, 0, 0
    for _, a in d.iterrows():
        i = int(a.id); txt = ans[key[i]]
        txt = txt if isinstance(txt, str) else json.dumps(txt)
        rv = result_for(EX[int(a.PMID)]).review(a.conclusion, llm=CallableLLM(lambda s, u, t=txt: t), rewrite=False, record=False)
        # a readable reply = it contained an "assertions" list (0.6.1 reports this as rv.read_ok)
        ok += not any(e.startswith(("the language model's reply had no", 'the language model could not be used')) for e in rv.errors)
        item_err += sum(e.startswith('an assertion read by the model was not usable') for e in rv.errors)
        bad = [v for v in rv.verdicts if v.assertion.polarity == 'asserted' and v.status in ('contradicted', 'not_established')]
        flag[i] = bool(bad)
        reader_detail[(m, i)] = [(v.assertion.kind, v.assertion.direction, v.assertion.polarity, v.status, v.assertion.quote) for v in rv.verdicts]
    print(f'{m}: replies with an assertions list {ok}/60; individual assertions not usable (reported): {item_err}')
    preds[f'Claude {m} reads, statwarrant decides'] = flag
    preds[f'Claude {m} reads, statwarrant decides, + rule conflicts'] = {i: flag[i] or preds['statwarrant rules (conflict)'][i] for i in IDS}
# What the model read in one spun conclusion, and statwarrant's verdicts:
i = int(d[d.abstract_type == 'spin'].iloc[3].id)
print(d[d.id == i].conclusion.iloc[0]); pd.DataFrame(reader_detail[('sonnet', i)], columns=['kind', 'direction', 'polarity', 'verdict', 'quote'])

haiku: replies with an assertions list 60/60; individual assertions not usable (reported): 0
sonnet: replies with an assertions list 60/60; individual assertions not usable (reported): 2
However, exploratory analyses revealed improvement in some secondary outcomes, such as fatigue. Further research with treatment A exploring physiologic effects in oncology symptom management may be warranted.


,kind,direction,polarity,verdict,quote
0,nonzero,positive,asserted,not_established,"However, exploratory analyses revealed improvement in some secondary outcomes, such as fatigue."


## 9. Scores: accuracy, sensitivity, specificity, Wilson and trial-pair bootstrap intervals

In [16]:
def wilson(k, n, z=1.959964):
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n); h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return p, c - h, c + h
pmid_of = {int(r.id): int(r.PMID) for _, r in d.iterrows()}
pairs = sorted(set(pmid_of.values())); by_pair = {p: [i for i in IDS if pmid_of[i] == p] for p in pairs}
def pair_boot(pred, B=10000, seed=1):
    rng = random.Random(seed); accs = []
    for _ in range(B):
        ids = [i for p in rng.choices(pairs, k=len(pairs)) for i in by_pair[p]]
        accs.append(sum(pred[i] == label[i] for i in ids) / len(ids))
    accs.sort(); return accs[int(0.025*B)], accs[int(0.975*B) - 1]
def score(name, pred):
    tp = sum(pred[i] and label[i] for i in IDS); tn = sum(not pred[i] and not label[i] for i in IDS)
    acc = wilson(tp + tn, 60); b = pair_boot(pred)
    return dict(method=name, accuracy=round(100*acc[0]), wilson_95=f'{100*acc[1]:.0f}-{100*acc[2]:.0f}',
                pair_bootstrap_95=f'{100*b[0]:.0f}-{100*b[1]:.0f}', spin_flagged=f'{tp}/30', neutral_passed=f'{tn}/30')
table = pd.DataFrame([score(n, p) for n, p in {**preds, **published}.items()]).sort_values('accuracy', ascending=False)
table.reset_index(drop=True)

,method,accuracy,wilson_95,pair_bootstrap_95,spin_flagged,neutral_passed
0,"Claude haiku, same inputs as statwarrant",100,94-100,100-100,30/30,30/30
1,"Claude opus, same inputs as statwarrant",100,94-100,100-100,30/30,30/30
2,"Claude sonnet, same inputs as statwarrant",100,94-100,100-100,30/30,30/30
3,"Claude opus, full abstract",100,94-100,100-100,30/30,30/30
4,"Claude opus, full abstract, one context",100,94-100,100-100,30/30,30/30
5,claude_3.5-sonnet (published),97,89-99,92-100,28/30,30/30
6,"Claude haiku, full abstract, one context",93,84-97,87-98,26/30,30/30
7,"Claude sonnet, full abstract, one context",93,84-97,87-98,26/30,30/30
8,"Claude sonnet, full abstract",92,82-96,85-98,25/30,30/30
9,"Claude haiku, full abstract",92,82-96,83-98,26/30,29/30


## 10. Paired comparisons with statwarrant's rules (exact McNemar)
`rules_right_only` = abstracts the rules got right and the other method got wrong.

In [17]:
base = preds['statwarrant rules (conflict)']
def mcnemar(other):
    ra = [base[i] == label[i] for i in IDS]; rb = [other[i] == label[i] for i in IDS]
    n01 = sum(x and not y for x, y in zip(ra, rb)); n10 = sum(y and not x for x, y in zip(ra, rb))
    return n01, n10, (1.0 if n01 + n10 == 0 else binomtest(n01, n01 + n10, 0.5).pvalue)
rows = []
for n, p in {**preds, **published}.items():
    if n == 'statwarrant rules (conflict)': continue
    a, b, pv = mcnemar(p); rows.append(dict(method=n, rules_right_only=a, other_right_only=b, p_value=round(pv, 4)))
pd.DataFrame(rows).sort_values('p_value')

,method,rules_right_only,other_right_only,p_value
2,"Claude haiku, same inputs as statwarrant",0,13,0.0002
7,"Claude opus, full abstract, one context",0,13,0.0002
6,"Claude opus, full abstract",0,13,0.0002
5,"Claude sonnet, same inputs as statwarrant",0,13,0.0002
8,"Claude opus, same inputs as statwarrant",0,13,0.0002
30,mistral_instruct7B (published),19,2,0.0002
20,gpt35 (published),18,2,0.0004
31,olmo2_instruct-13B (published),18,2,0.0004
15,biomistral7B (published),19,3,0.0009
17,claude_3.5-sonnet (published),1,12,0.0034


## 10b. A wording cue in the benchmark
Every neutral rewrite contains "was not more effective than"; no spun abstract does. A one-line rule using only that phrase:

In [18]:
phrase_rule = {int(r.id): 'was not more effective than' not in r.abstract.replace('\u2010', '-') for _, r in d.iterrows()}
phrase_concl = {int(r.id): 'was not more effective than' not in r.conclusion for _, r in d.iterrows()}
print(score('phrase rule, full abstract', phrase_rule)); print(score('phrase rule, conclusion only', phrase_concl))

{'method': 'phrase rule, full abstract', 'accuracy': 100, 'wilson_95': '94-100', 'pair_bootstrap_95': '100-100', 'spin_flagged': '30/30', 'neutral_passed': '30/30'}
{'method': 'phrase rule, conclusion only', 'accuracy': 100, 'wilson_95': '94-100', 'pair_bootstrap_95': '100-100', 'spin_flagged': '30/30', 'neutral_passed': '30/30'}


## 10c. Comparisons by trial pair (the two versions of a trial are not independent)
Per trial pair: number of correct answers (0, 1 or 2). Sign test on the 30 pair differences, and a bootstrap of the accuracy difference resampling pairs.

In [19]:
def pair_test(a, b, B=20000, seed=3):
    ca = {p: sum(a[i] == label[i] for i in by_pair[p]) for p in pairs}
    cb = {p: sum(b[i] == label[i] for i in by_pair[p]) for p in pairs}
    diff = [cb[p] - ca[p] for p in pairs]
    pos, neg = sum(x > 0 for x in diff), sum(x < 0 for x in diff)
    sp = 1.0 if pos + neg == 0 else binomtest(pos, pos + neg, 0.5).pvalue
    rng = random.Random(seed); bs = sorted(sum(rng.choices(diff, k=len(diff))) / 60 for _ in range(B))
    return pos, neg, sp, sum(diff) / 60, bs[int(0.025 * B)], bs[int(0.975 * B) - 1]
rows = []
for n, p in {**preds, **published, 'phrase rule': phrase_rule}.items():
    if n == 'statwarrant rules (conflict)': continue
    pos, neg, sp, md_, lo, hi = pair_test(base, p)
    rows.append(dict(method=n, pairs_better_than_rules=pos, pairs_worse=neg, sign_test_p=round(sp, 4),
                     accuracy_difference=f'{100*md_:+.0f} ({100*lo:+.0f} to {100*hi:+.0f})'))
pd.DataFrame(rows).sort_values('sign_test_p')

,method,pairs_better_than_rules,pairs_worse,sign_test_p,accuracy_difference
14,biomedgpt7B (published),1,18,0.0001,-28 (-38 to -18)
31,olmo2_instruct-13B (published),1,17,0.0001,-27 (-37 to -17)
30,mistral_instruct7B (published),1,18,0.0001,-28 (-38 to -18)
20,gpt35 (published),1,17,0.0001,-27 (-37 to -17)
25,llama2_chat-7B (published),1,18,0.0001,-28 (-38 to -18)
34,openbiollm-8B (published),1,18,0.0001,-28 (-38 to -18)
15,biomistral7B (published),2,18,0.0004,-27 (-37 to -15)
2,"Claude haiku, same inputs as statwarrant",12,0,0.0005,+22 (+12 to +32)
6,"Claude opus, full abstract",12,0,0.0005,+22 (+12 to +32)
8,"Claude opus, same inputs as statwarrant",12,0,0.0005,+22 (+12 to +32)


## 10d. Claim by claim: was each conflict about the primary result?
`data/primary_linked.csv` classifies every conflict (one rater). Restricted to conflicts on claims about the primary result:

In [20]:
pl = pd.read_csv('data/primary_linked.csv'); display(pl)
linked = {(int(r.pmid), r.version): r.linked_to_primary == 'yes' for r in pl.itertuples()}
restricted = {i: base[i] and linked.get((pmid_of[i], 'spin' if label[i] else 'no_spin'), False) for i in IDS}
print(score('rules, conflicts on primary-result claims only', restricted))

,pmid,version,conflict_about,linked_to_primary,note
0,9093724,spin,significant increase of antitumor effect,no,response rate (secondary)
1,11261827,spin,efficient and well-tolerated agent,yes,overall efficacy of treatment A
2,12177098,spin,at least as effective ... slightly favoring,yes,overall efficacy (primary TTP)
3,15947110,spin,significantly higher reduction in Ki67,no,Ki67 (secondary)
4,16314619,spin,may improve response rates and survival,yes,survival is the primary outcome
5,16504757,spin,significantly better locoregional control,no,local control (secondary)
6,17134892,spin,had no detectable effect on overall survival,yes,primary; an over-claim of no effect
7,17173959,spin,"clinically meaningful increase in the detection of CIN 2,3",yes,primary (detection rate)
8,18794551,spin,effective ... significant early improvement in time to tumor progression,yes,primary (PFS)
9,18955454,spin,significantly improved clinical outcomes in HER-2-positive patients,no,subgroup


{'method': 'rules, conflicts on primary-result claims only', 'accuracy': 63, 'wilson_95': '51-74', 'pair_bootstrap_95': '57-72', 'spin_flagged': '8/30', 'neutral_passed': '30/30'}


## 11. Why the rules miss: every spun conclusion without a conflict

In [21]:
rules[(rules.label == 'spin') & (~rules.flag)][['PMID', 'level', 'to_verify', 'conclusion']]

,PMID,level,to_verify,conclusion
56,10547391,check,nonsignificance; direction,"Treatment A of women with breast cancer for 5 years appears to have no statistically significant effect on the incidence of second breast malignancies overall, although a possible benefit was dete..."
44,10637238,check,direction; unattached-negation,"In patients with potentially hormone-sensitive metastatic breast cancer, treatment A + chemotherapy prolongs TTF for ER-positive patients without improving overall survival."
18,16148021,check,unattached-negation; significance,"The present study failed to demonstrate a significant difference in DFS or OS between the two treatment groups. However, our study has shown clearly that treatment A can be safely incorporated to ..."
38,17179098,check,subgroup; unattached-negation,Treatment A + CAF was well tolerated and is suggested to have efficacy in patients who had not received prior therapy.
52,17264336,none,,This treatment protocol remains an appealing clinical option for many women with operable breast cancer at a high risk of recurrence. Combination treatments with new drugs for breast cancer are wa...
2,17530429,check,direction; percentage; percentage; subgroup; subgroup,An even higher reduction of 80% and 84% in relapse and death was seen for the patients who had received the full treatment A dose. We identified a subgroup of patients with intermediate proliferat...
20,20153039,check,no-effect; importance; subgroup; direction,"Although treatment A does not improve overall or progression-free survival compared with comparator B, treatment A might still have a role because it can produce remissions that are of clinically ..."
48,20530276,check,direction; unattached-negation; causal,"This study confirmed a previous trial demonstrating improved PFS and response for the treatment A compared with comparator B alone, although this did not result in improved survival."
22,20564068,check,subgroup; unattached-negation,Treatment A may be useful in controlling cancer-related fatigue in patients who present with severe fatigue but is not useful in patients with mild or moderate fatigue.
8,20673585,check,direction,Female patients performed better than male patients. Better survival obtained among female AML patients needs to be validated within the context of cytogenetic analysis.


In [22]:
# The two false alarms on neutral conclusions:
rules[(rules.label != 'spin') & (rules.flag)][['PMID', 'conflicts', 'conclusion']]

,PMID,conflicts,conclusion
9,20673585,"p-value: ""p=0.38""",The three-year OS rates were not statistically significantly different between treatment A and control arm (25.6 +/- 5.1% vs. 31.8 +/- 5.6%; p=0.38. Treatment A as an adjunct to cytarabine and ida...
7,21399726,"p-value: ""p<0.01""; p-value: ""p=0.05""; p-value: ""p=0.65""; p-value: ""p=0.75""",For the BFI at week 8 the treatment A and placebo arm scored respectively for fatigue now 22.1 vs. 10.5; p<0.01; for usual fatigue 19.4 vs. 10.0; p=0.05; for worst fatigue 14.8 vs. 12.4; p=0.65; a...


## 12. Optional: re-run a Claude model yourself
Needs `pip install anthropic` and an `ANTHROPIC_API_KEY`. Set `RUN = True` and the exact model id.
This uses one API call per abstract (the recorded runs above used 30 abstracts per context).

In [23]:
RUN = False
MODEL = 'claude-...'   # put the exact model id here
PROMPT = '''Spin or misrepresentation of study findings can be used to influence, positively, the interpretation of statistically nonsignificant randomized controlled trials (RCTs), for example, by emphasizing the apparent benefit of a secondary outcome or findings from a subgroup of patients.
Does the following clinical trial abstract contain spin (yes/no)?
Answer only with 'yes' or 'no'. Do not provide any explanations.

Abstract: {ABSTRACT}'''
if RUN:
    import anthropic
    client = anthropic.Anthropic()
    mine = {}
    for _, a in d.iterrows():
        msg = client.messages.create(model=MODEL, max_tokens=5, messages=[{'role': 'user', 'content': PROMPT.format(ABSTRACT=a.abstract)}])
        mine[int(a.id)] = msg.content[0].text.strip().lower().startswith('yes')
    print(score(MODEL, mine))

## Limitations (read before citing any number)
* 60 abstracts, 30 trials, one field (oncology), all with non-significant primary outcomes.
* The abstracts (2014) and MedLitSpin (2025) are public: the language models may have seen them in training.
* The neutral conclusions share one template ("Treatment A was not more effective than ..."), which makes the task easy for a reader of wording.
* The primary results were extracted by an AI system and checked by a second AI extraction, not by a person.
* The Claude answers were produced through an agent framework, 30 abstracts per context, not one API call each.